In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score

In [5]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [6]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [7]:
categorical = list(df.select_dtypes(exclude='number').columns)
numerical = [c for c in df.select_dtypes(include='number').columns if c != 'converted']
print("categorical:", categorical)
print("numerical:  ", numerical)

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical:   ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [8]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()   # should print 0

np.int64(0)

In [9]:
print(df.industry.mode()[0])
df.industry.value_counts()

technology


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [10]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [11]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f"{a:26s} {b:20s} {corr.loc[a, b]:.4f}")

interaction_count          lead_score           0.9157
number_of_courses_viewed   lead_score           0.7572
number_of_courses_viewed   interaction_count    0.7216
annual_income              interaction_count    0.1228


In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print(len(df_train), len(df_val), len(df_test))
print('converted' in df_train.columns)

3000 1000 1000
False


In [15]:
def mi_score(col):
    return mutual_info_score(col, y_train)

mi = df_train[categorical].apply(mi_score)
mi.round(4).sort_values(ascending=False)


lead_source          0.0343
employment_status    0.0190
industry             0.0020
location             0.0010
dtype: float64

In [19]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    return float(((y_pred >= 0.5) == y_val).mean())

features = categorical + numerical
base_acc = train_and_score(features)

print("accuracy:", round(base_acc, 2))

accuracy: 0.65


In [26]:
diffs = {}
for f in features:
    subset = [x for x in features if x != f]
    acc = train_and_score(subset)
    diffs[f] = base_acc - acc

diffs = pd.Series(diffs)
diffs.sort_values()

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [34]:
options = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
diffs[options].sort_values()

number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [23]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = {}
for C in C_values:
    results[C] = round(train_and_score(features, C=C), 3)
    print(C, results[C])

best_acc = max(results.values())
best_C = min(C for C, acc in results.items() if acc == best_acc)
print("best accuracy:", best_acc, "-> C =", best_C)

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
best accuracy: 0.645 -> C = 0.001
